# Dynamic Scene Object / Skeleton / Gaze Viewer

This notebook is a dynamic companion to `05_scene_object_gaze_viewer.ipynb`.
It does not replace the static viewer. Use the play control or focus-frame slider
to update the selected frame and inspect dynamic object boxes, skeleton pose,
gaze ray, gaze point, and ray-box hit object over time.


In [1]:
from pathlib import Path
import sys
import importlib

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

import visualization.scene_object_viewer as scene_object_viewer
importlib.reload(scene_object_viewer)

from visualization.scene_object_viewer import (
    build_scene_object_gaze_figure,
    discover_scene_viewer_sequence_ids,
    load_scene_viewer_data,
)

REPORTS_DIR = Path("/mnt/d/SparseGaze/ADT-Gaze-structured")
sequence_ids = discover_scene_viewer_sequence_ids(REPORTS_DIR)
sequence_ids[:3], len(sequence_ids)


(['Apartment_release_decoration_skeleton_seq131_M1292',
  'Apartment_release_decoration_skeleton_seq132_M1292',
  'Apartment_release_decoration_skeleton_seq133_M1292'],
 34)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

_data_cache = {}
_last_rendered_focus = None

def get_data(sequence_id):
    if sequence_id not in _data_cache:
        _data_cache[sequence_id] = load_scene_viewer_data(REPORTS_DIR, sequence_id)
    return _data_cache[sequence_id]

sequence_dropdown = widgets.Dropdown(
    options=sequence_ids,
    value=sequence_ids[0],
    description="sequence",
    layout=widgets.Layout(width="900px"),
)
start_input = widgets.IntText(value=0, description="start")
end_input = widgets.IntText(value=180, description="end")
frame_step_input = widgets.IntText(value=5, description="frame step")
context_radius_input = widgets.IntText(value=30, description="context")
window_stride_input = widgets.IntText(value=5, description="draw stride")
max_static_input = widgets.IntText(value=120, description="max static")

focus_slider = widgets.IntSlider(value=0, min=0, max=179, step=5, description="focus")
play = widgets.Play(value=0, min=0, max=179, step=5, interval=180, description="play")
widgets.jslink((play, "value"), (focus_slider, "value"))

category_text = widgets.Text(value="", description="categories", layout=widgets.Layout(width="520px"))
exclude_category_text = widgets.Text(value="shelter", description="exclude", layout=widgets.Layout(width="520px"))

show_static_checkbox = widgets.Checkbox(value=True, description="static boxes")
show_dynamic_checkbox = widgets.Checkbox(value=True, description="dynamic boxes")
show_skeleton_checkbox = widgets.Checkbox(value=True, description="skeleton")
show_skeleton_traj_checkbox = widgets.Checkbox(value=True, description="root path")
show_head_checkbox = widgets.Checkbox(value=True, description="head path")
show_gaze_rays_checkbox = widgets.Checkbox(value=True, description="gaze rays")
show_gaze_points_checkbox = widgets.Checkbox(value=True, description="gaze points")
show_hit_point_checkbox = widgets.Checkbox(value=True, description="hit point")
show_hit_object_checkbox = widgets.Checkbox(value=True, description="hit object outline")
auto_render_checkbox = widgets.Checkbox(value=True, description="auto render")

gaze_length_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="ray len",
    readout_format=".1f",
)
gaze_scale_dropdown = widgets.Dropdown(
    options=["fixed", "depth"],
    value="fixed",
    description="ray scale",
)
render_button = widgets.Button(description="Render current", button_style="primary")
status = widgets.HTML(
    value="Ready. Playback updates the view because <b>auto render</b> is on. "
    "If the notebook itself spins on open, restart the kernel and clear widget outputs."
)
output = widgets.Output()

def configure_range(_=None):
    data = get_data(sequence_dropdown.value)
    start = max(0, min(int(start_input.value), len(data.frames) - 1))
    end = min(max(int(end_input.value), start + 1), len(data.frames))
    step = max(1, int(frame_step_input.value))
    focus_slider.min = start
    focus_slider.max = end - 1
    focus_slider.step = step
    focus_slider.value = min(max(int(focus_slider.value), start), end - 1)
    play.min = start
    play.max = end - 1
    play.step = step
    play.value = focus_slider.value
    status.value = f"Selected range: {start}-{end - 1}. Focus: {focus_slider.value}. Auto render is {'on' if auto_render_checkbox.value else 'off'}."

def render(_=None):
    global _last_rendered_focus
    with output:
        output.clear_output(wait=True)
        data = get_data(sequence_dropdown.value)
        focus = int(focus_slider.value)
        radius = max(0, int(context_radius_input.value))
        start = max(int(start_input.value), focus - radius)
        end = min(int(end_input.value), focus + radius + 1, len(data.frames))
        max_static = None if max_static_input.value <= 0 else int(max_static_input.value)
        status.value = f"Rendering frame {focus} with context {start}-{end - 1}..."
        fig = build_scene_object_gaze_figure(
            data,
            start_frame=start,
            end_frame=end,
            stride=max(1, int(window_stride_input.value)),
            focus_frame=focus,
            show_static_objects=show_static_checkbox.value,
            show_dynamic_objects=show_dynamic_checkbox.value,
            max_static_objects=max_static,
            category_filter=category_text.value,
            exclude_category_filter=exclude_category_text.value,
            show_skeleton=show_skeleton_checkbox.value,
            show_skeleton_trajectory=show_skeleton_traj_checkbox.value,
            show_head_trajectory=show_head_checkbox.value,
            show_gaze_rays=show_gaze_rays_checkbox.value,
            show_gaze_points=show_gaze_points_checkbox.value,
            show_object_hits=show_hit_point_checkbox.value,
            show_hit_object=show_hit_object_checkbox.value,
            gaze_ray_length_m=gaze_length_slider.value,
            gaze_scale_mode=gaze_scale_dropdown.value,
        )
        display(fig)
        _last_rendered_focus = focus
        status.value = f"Rendered frame {focus}. Move focus and render again, or enable auto render for playback."

def maybe_auto_render(change=None):
    if auto_render_checkbox.value:
        render()
    else:
        status.value = f"Focus: {focus_slider.value}. Auto render is off; click <b>Render current</b> to update the 3D view."

for widget in [sequence_dropdown, start_input, end_input, frame_step_input]:
    widget.observe(configure_range, names="value")
focus_slider.observe(maybe_auto_render, names="value")
render_button.on_click(render)

configure_range()

controls = widgets.VBox([
    sequence_dropdown,
    widgets.HBox([start_input, end_input, frame_step_input, context_radius_input, window_stride_input, max_static_input]),
    widgets.HBox([play, focus_slider]),
    widgets.HBox([category_text, exclude_category_text]),
    widgets.HBox([gaze_scale_dropdown, gaze_length_slider]),
    widgets.HBox([show_static_checkbox, show_dynamic_checkbox, show_skeleton_checkbox, show_skeleton_traj_checkbox]),
    widgets.HBox([show_head_checkbox, show_gaze_rays_checkbox, show_gaze_points_checkbox, show_hit_point_checkbox, show_hit_object_checkbox, auto_render_checkbox]),
    render_button,
    status,
])

display(controls, output)


Output()

## Notes

- `start` / `end`: global frame range available to the play control.
- `frame step`: frame increment for play and the focus slider.
- `context`: number of neighboring frames drawn before/after the current focus frame for trajectories and gaze rays.
- `draw stride`: subsampling inside the rendered context window; larger values draw fewer rays/points and render faster.
- `max static`: maximum number of static object boxes to draw; `0` means no cap.
- `categories`: optional comma-separated object categories to include. Empty means all categories.
- `exclude`: comma-separated categories to hide. Default `shelter` removes the room-envelope box.
- `ray scale`: `fixed` draws fixed-length gaze rays for direction comparison; `depth` scales rays by ADT gaze depth.
- `ray len`: fixed ray length in meters when `ray scale=fixed`.
- `hit point`: shows the current ray-box intersection point.
- `hit object outline`: adds an outline to the object cuboid intersected by the current gaze ray. This is a ray-box hit cue, not confirmed attention/fixation.
- `auto render`: when enabled, slider/play changes rebuild the 3D figure. Disable it if the notebook UI becomes slow, then use `Render current`.
- If the notebook keeps spinning before any meaningful render, restart the kernel and rerun the two code cells. The notebook file is saved without outputs to avoid loading stale widget state.
